In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('..'))

import cv2
import time
import numpy as np
import pandas as pd
import kagglehub
import glob
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.models as models
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from utils.datasets import BusbraDataset
from utils.metric import BCEDiceLoss
from utils.training import train

# Hyper-parameters

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_RUNS = 5
EPOCHS = 80
LR = 1e-4
WEIGHT_DECAY = 5e-4
DROPOUT_P = 0.3

torch.backends.cudnn.benchmark = True
print(f"Using device: {DEVICE}")

# Load Dataset

In [ ]:
path = kagglehub.dataset_download("orvile/bus-bra-a-breast-ultrasound-dataset")
csv_candidates = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
img_candidates = glob.glob(os.path.join(path, '**', 'Images'), recursive=True)
mask_candidates = glob.glob(os.path.join(path, '**', 'Masks'), recursive=True)
csv_path = csv_candidates[0]
images_root = img_candidates[0]
masks_root = mask_candidates[0]

df_meta = pd.read_csv(csv_path)
entries = []
for _, row in df_meta.iterrows():
    bid = str(row['ID'])
    img_f = os.path.join(images_root, f"{bid}.png")
    mask_f = os.path.join(masks_root, bid.replace('bus_', 'mask_') + '.png')
    if os.path.exists(img_f) and os.path.exists(mask_f):
        entries.append((row['Pathology'], img_f, mask_f))

df = pd.DataFrame(entries, columns=['label', 'image_path', 'mask_path'])
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'])
print(f"Found {len(df)} pairs. Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
train_transform = A.Compose([
    A.Resize(*IMG_SIZE), A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(0.05, 0.1, 15, p=0.5),
    A.ElasticTransform(alpha=50, sigma=50*0.05, p=0.3),
    A.RandomBrightnessContrast(p=0.8), A.GaussNoise(p=0.3),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0), max_pixel_value=255.0),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0), max_pixel_value=255.0),
    ToTensorV2()
])

train_ds = BusbraDataset(train_df, IMG_SIZE, train_transform)
val_ds = BusbraDataset(val_df, IMG_SIZE, val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print("Data ready.")

# Ablation Components

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channel, channel//reduction, 1), nn.ReLU(True),
            nn.Conv2d(channel//reduction, channel, 1), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(self.pool(x))

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(True)
    def forward(self, x):
        return self.relu(self.bn(self.pw(self.dw(x))))

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, layers=1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads)
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
    def forward(self, x):
        b, c, h, w = x.shape
        xf = x.flatten(2).permute(2, 0, 1)
        xf = self.encoder(xf)
        return xf.permute(1, 2, 0).view(b, c, h, w)

class MultiScaleFusion(nn.Module):
    def __init__(self, in_ch, out_ch, scales=[1,2,4]):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Upsample(scale_factor=s, mode='bilinear', align_corners=False),
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(True)
            ) for s in scales])
        self.se = SEBlock(out_ch * len(scales))
        self.project = nn.Conv2d(out_ch * len(scales), out_ch, 1)
    def forward(self, x):
        feats = [b(x) for b in self.branches]
        mh = max(f.shape[2] for f in feats)
        mw = max(f.shape[3] for f in feats)
        feats = [F.interpolate(f, size=(mh,mw), mode='bilinear', align_corners=False) for f in feats]
        return self.project(self.se(torch.cat(feats, dim=1)))

In [ ]:
class MobileNetV2_UNet_Ablation(nn.Module):
    """MobileNetV2-UNet with toggleable bottleneck components."""
    def __init__(self, out_channels=1, use_se=False, use_transformer=False, use_msf=False):
        super().__init__()
        self.use_se = use_se
        self.use_transformer = use_transformer
        self.use_msf = use_msf
        net = models.mobilenet_v2(pretrained=True)
        self.encoder = net.features
        idx = [1, 3, 6, 13]
        ch = [self.encoder[i].out_channels for i in idx]
        self.bottle = nn.Conv2d(net.features[-1].out_channels, ch[-1], 1)
        if use_se: self.se = SEBlock(ch[-1])
        if use_transformer: self.trans = TransformerBlock(ch[-1])
        if use_msf: self.msf = MultiScaleFusion(ch[-1], ch[-1])
        self.dropout = nn.Dropout2d(DROPOUT_P)
        self.refine = nn.Conv2d(ch[-1], ch[-1], 1)
        self.up_convs, self.dec_convs = nn.ModuleList(), nn.ModuleList()
        for i in range(len(ch)-1, 0, -1):
            self.up_convs.append(nn.ConvTranspose2d(ch[i], ch[i-1], 2, 2))
            self.dec_convs.append(DepthwiseSeparableConv(ch[i-1]*2, ch[i-1]))
        self.final = nn.Conv2d(ch[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for i, layer in enumerate(self.encoder):
            x = layer(x)
            if i in [1,3,6,13]: skips.append(x)
        x = self.bottle(x)
        if self.use_se: x = self.se(x)
        if self.use_transformer: x = self.trans(x)
        if self.use_msf: x = self.msf(x)
        x = self.dropout(x)
        x = F.relu(self.refine(x))
        for up, dec, skip in zip(self.up_convs, self.dec_convs, reversed(skips[:-1])):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        x = F.interpolate(x, size=IMG_SIZE, mode='bilinear', align_corners=False)
        return self.final(x)

# Run Ablation Experiments

Four configurations tested progressively:
- **Base**: MobileNetV2 + Decoder only
- **Base + SE**: Add SE block
- **Base + SE + Trans**: Add transformer layer
- **Base + SE + Trans + MSF**: Add multi-scale fusion (full variant)

In [ ]:
configs = [
    ("Base (proposed)",          False, False, False),
    ("Base + SE",               True,  False, False),
    ("Base + SE + Trans",       True,  True,  False),
    ("Base + SE + Trans + MSF", True,  True,  True),
]

all_results = []
for name, use_se, use_trans, use_msf in configs:
    print(f"\n{'='*60}")
    print(f"  ABLATION: {name}")
    print(f"  SE={use_se}, Trans={use_trans}, MSF={use_msf}")
    print(f"{'='*60}")

    run_dices, run_precs, run_mious = [], [], []
    for run in range(1, NUM_RUNS + 1):
        print(f"\n--- Run {run}/{NUM_RUNS} ---")
        model = MobileNetV2_UNet_Ablation(
            use_se=use_se, use_transformer=use_trans, use_msf=use_msf
        ).to(DEVICE)
        criterion = BCEDiceLoss(weight_bce=0.5, weight_dice=0.5)
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

        history, best_iou, best_dice, best_prec = train(
            model, DEVICE, train_loader, val_loader, criterion,
            optimizer, scheduler, num_epochs=EPOCHS,
            save_path=f'ablation_{name.replace(" ", "_")}_run{run}.pth'
        )
        run_dices.append(best_dice)
        run_precs.append(best_prec)
        run_mious.append(best_iou)
        print(f"  Run {run} best -> Dice:{best_dice:.4f} Prec:{best_prec:.4f} mIoU:{best_iou:.4f}")

    da = np.array(run_dices)
    pa = np.array(run_precs)
    ma = np.array(run_mious)
    params = sum(p.numel() for p in model.parameters()) / 1e6

    print(f"\n  {name} SUMMARY:")
    print(f"  Dice : {da.mean()*100:.2f} +/- {da.std()*100:.2f} %")
    print(f"  Prec : {pa.mean()*100:.2f} +/- {pa.std()*100:.2f} %")
    print(f"  mIoU : {ma.mean()*100:.2f} +/- {ma.std()*100:.2f} %")
    print(f"  Params: {params:.2f}M")

    all_results.append({'name': name, 'dice': da.mean()*100, 'dice_std': da.std()*100,
                        'prec': pa.mean()*100, 'prec_std': pa.std()*100,
                        'miou': ma.mean()*100, 'miou_std': ma.std()*100, 'params': params})

# Summary

In [ ]:
print("\n" + "="*70)
print("  ABLATION STUDY RESULTS")
print("="*70)
print(f"  {'Configuration':<30s}  {'Dice (%)':<18s}  {'mIoU (%)':<18s}  {'Params'}")
print("  " + "-"*66)
for r in all_results:
    print(f"  {r['name']:<30s}  {r['dice']:.2f} +/- {r['dice_std']:.2f}    "
          f"{r['miou']:.2f} +/- {r['miou_std']:.2f}    {r['params']:.2f}M")
print("="*70)